# Liperty VSR Training Notebook (Parallel & GDrive Persistent)
This version of the notebook is optimized for **speed** (simultaneous downloads) and **persistence** (Google Drive storage).

**Features:**
- Uses `aria2c` for high-speed parallel downloads.
- Persistent storage on Google Drive (`MyDrive/LipertyData`).
- Automated setup and LoRA initialization.

In [4]:
import os

def check_path(path):
    return "Exists" if os.path.exists(path) else "Missing"

paths = {
    "Repository": "/content/Liperty",
    "LRS2 Archive": "/content/Liperty/data/LRS2-2Mix/lrs2.tar.gz",
    "VVAD-LRS3 Archive": "/content/Liperty/data/VVAD-LRS3/vvadlrs3.zip",
    "Checkpoints Dir": "/content/Liperty/data/checkpoints"
}

print("--- Progress Report ---")
for name, path in paths.items():
    status = check_path(path)
    print(f"{name}: {status}")

if os.path.exists("/content/Liperty/data/checkpoints"):
    checkpoints = os.listdir("/content/Liperty/data/checkpoints")
    print(f"Checkpoints found: {checkpoints if checkpoints else 'None yet'}")

--- Progress Report ---
Repository: Missing
LRS2 Archive: Missing
VVAD-LRS3 Archive: Missing
Checkpoints Dir: Missing


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

gdrive_path = '/content/drive/MyDrive/LipertyData'

print(f"--- Checking GDrive: {gdrive_path} ---")
if os.path.exists(gdrive_path):
    contents = os.listdir(gdrive_path)
    print(f"Contents found: {contents if contents else 'Empty folder'}")

    # Check for specific subdirectories if they exist
    for sub in ['LRS2-2Mix', 'VVAD-LRS3', 'checkpoints']:
        sub_path = os.path.join(gdrive_path, sub)
        if os.path.exists(sub_path):
            print(f"{sub}: Exists (contains {len(os.listdir(sub_path))} items)")
        else:
            print(f"{sub}: Missing")
else:
    print("LipertyData folder not found on GDrive.")

--- Checking GDrive: /content/drive/MyDrive/LipertyData ---
Contents found: ['LipertyData', 'checkpoints']
LRS2-2Mix: Missing
VVAD-LRS3: Missing
checkpoints: Exists (contains 0 items)


In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Define Persistence Paths
import os
gdrive_data_root = '/content/drive/MyDrive/LipertyData'
!mkdir -p {gdrive_data_root}

# 3. Clone Repository
repo_dir = '/content/Liperty'
if not os.path.exists(repo_dir):
    !git clone https://github.com/HereLiesAz/Liperty.git {repo_dir}

%cd {repo_dir}

# 4. Symlink GDrive Data to the Repo
!ln -snf {gdrive_data_root} /content/Liperty/data

# 5. Install Dependencies
!apt-get install -y aria2
!chmod +x setup_libs.sh
!./setup_libs.sh
!pip install datasets transformers mediapipe opencv-python onnx onnx-tf tensorflow torch

Mounted at /content/drive
Cloning into '/content/Liperty'...
remote: Enumerating objects: 3374, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 3374 (delta 23), reused 20 (delta 7), pack-reused 3296 (from 3)
Receiving objects: 100% (3374/3374), 377.85 MiB | 21.91 MiB/s, done.
Resolving deltas: 100% (1370/1370), done.
Updating files: 100% (1159/1159), done.
/content/Liperty
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libaria2-0 libc-ares2
The following NEW packages will be installed:
  aria2 libaria2-0 libc-ares2
0 upgraded, 3 newly installed, 0 to remove and 42 not upgraded.
Need to get 1,513 kB of archives.
After this operation, 5,441 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libc-ares2 amd64 1.18.1-1ubuntu0.22.04.3 [45.1 kB]
Get:2 http://archive.ubu

In [19]:
import os
checkpoint_dir = '/content/drive/MyDrive/LipertyData/LipertyData/checkpoints'

if os.path.exists(checkpoint_dir):
    files = os.listdir(checkpoint_dir)
    print(f'Found {len(files)} files in {checkpoint_dir}:')
    for f in sorted(files):
        size = os.path.getsize(os.path.join(checkpoint_dir, f)) / (1024 * 1024)
        print(f'- {f} ({size:.2f} MB)')
else:
    print(f'Checkpoint directory {checkpoint_dir} not found.')

Found 1 files in /content/drive/MyDrive/LipertyData/LipertyData/checkpoints:
- best_model.pth (704.25 MB)


In [22]:
import os
import time
from datetime import datetime

checkpoint_path = '/content/drive/MyDrive/LipertyData/LipertyData/checkpoints/best_model.pth'

if os.path.exists(checkpoint_path):
    stats = os.stat(checkpoint_path)
    last_mod = datetime.fromtimestamp(stats.st_mtime).strftime('%Y-%m-%d %H:%M:%S')
    size_mb = stats.st_size / (1024 * 1024)
    print(f'Current Checkpoint Status:')
    print(f'- Last Updated: {last_mod}')
    print(f'- File Size: {size_mb:.2f} MB')
else:
    print('Checkpoint file not found yet. Training may still be in early stages.')

Current Checkpoint Status:
- Last Updated: 2026-03-19 05:53:18
- File Size: 704.25 MB


In [26]:
import os
# Check for common log locations or the root for new log files
logs = [f for f in os.listdir('/content/Liperty') if f.endswith('.log') or 'train' in f.lower()]
if logs:
    print(f'Found log files: {logs}')
    for log in logs:
        print(f'\n--- Last 10 lines of {log} ---')
        !tail -n 10 /content/Liperty/{log}
else:
    print('No dedicated log files found in /content/Liperty. Training progress should be visible in the output of the training cell.')

Found log files: ['Copy_of_liperty_vsr_training.ipynb']

--- Last 10 lines of Copy_of_liperty_vsr_training.ipynb ---
      "display_name": "Python 3",
      "name": "python3"
    },
    "language_info": {
      "name": "python"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 0
}

In [27]:
import os
!ls -R /content/Liperty

/content/Liperty:
AGENTS.md			    gradle		 setup_libs.sh
app				    gradle.properties	 tools
build.gradle.kts		    gradlew		 uncleshadow.ipynb
CLAUDE.md			    gradlew.bat		 VALLR
Copy_of_liperty_vsr_training.ipynb  LICENSE		 version.properties
data				    README.md
docs				    settings.gradle.kts

/content/Liperty/app:
build.gradle.kts  src

/content/Liperty/app/src:
androidTest  main  test

/content/Liperty/app/src/androidTest:
java

/content/Liperty/app/src/androidTest/java:
com

/content/Liperty/app/src/androidTest/java/com:
HereLiesAz

/content/Liperty/app/src/androidTest/java/com/HereLiesAz:
liperty

/content/Liperty/app/src/androidTest/java/com/HereLiesAz/liperty:
IntegrationTest.kt  PrivacyTest.kt

/content/Liperty/app/src/main:
AndroidManifest.xml  assets  cpp  ic_launcher-playstore.png  java  res

/content/Liperty/app/src/main/assets:
face_landmarker.task  ssr_model.tflite	      vsr_lora_model.tflite
hand_landmarker.task  tramba_model.tflite     vsr_model.tflite
homophones.jso

In [5]:
!ls -R /content/Liperty

/content/Liperty:
AGENTS.md			    gradle		 setup_libs.sh
app				    gradle.properties	 tools
build.gradle.kts		    gradlew		 uncleshadow.ipynb
CLAUDE.md			    gradlew.bat		 VALLR
Copy_of_liperty_vsr_training.ipynb  LICENSE		 version.properties
data				    README.md
docs				    settings.gradle.kts

/content/Liperty/app:
build.gradle.kts  src

/content/Liperty/app/src:
androidTest  main  test

/content/Liperty/app/src/androidTest:
java

/content/Liperty/app/src/androidTest/java:
com

/content/Liperty/app/src/androidTest/java/com:
HereLiesAz

/content/Liperty/app/src/androidTest/java/com/HereLiesAz:
liperty

/content/Liperty/app/src/androidTest/java/com/HereLiesAz/liperty:
IntegrationTest.kt  PrivacyTest.kt

/content/Liperty/app/src/main:
AndroidManifest.xml  assets  cpp  ic_launcher-playstore.png  java  res

/content/Liperty/app/src/main/assets:
face_landmarker.task  ssr_model.tflite	      vsr_lora_model.tflite
hand_landmarker.task  tramba_model.tflite     vsr_model.tflite
homophones.jso

## Training & Fine-Tuning

In [ ]:
import kagglehub
import os

# Target directory on Google Drive
gdrive_target = '/content/drive/MyDrive/LipertyData/LipertyData/datasets'
os.makedirs(gdrive_target, exist_ok=True)

print(f'Starting download to Google Drive: {gdrive_target}')
# Downloading to Drive to bypass local disk limits
path = kagglehub.dataset_download("lipreadingankaya/lip-reading-dataset-word", output_dir=gdrive_target)

print(f'Dataset downloaded and extracted at: {path}')

In [76]:
import os

dataset_file = '/content/Liperty/VALLR/Data/dataset.py'
with open(dataset_file, 'r') as f:
    content = f.read()

# Add cv2 import if it's not there
if 'import cv2' not in content:
    content = 'import cv2\n' + content

with open(dataset_file, 'w') as f:
    f.write(content)

print("Successfully added 'import cv2' to dataset.py.")

Successfully added 'import cv2' to dataset.py.


In [73]:
import inspect
import sys
import os

# Ensure the VALLR path is in sys.path
repo_path = '/content/Liperty/VALLR'
if repo_path not in sys.path:
    sys.path.append(repo_path)

from Data.dataset import load_and_preprocess_video

# Get the source code of the function
source = inspect.getsource(load_and_preprocess_video)
print(f'--- Source of load_and_preprocess_video ---\n{source}')

--- Source of load_and_preprocess_video ---
def load_and_preprocess_video(video_path, num_frames):
    try:
        vr = VideoReader(video_path, ctx=cpu(0), num_threads=4)
    except Exception as e:
        print(f"Error loading video: {video_path}. Error: {e}")
        return None

    frame_count = len(vr)
    sample_indices = np.linspace(0, frame_count - 1, num_frames).astype(int)

    frames = []
    for idx in sample_indices:
        frame = vr[idx].asnumpy()
        frames.append(frame)

    if len(frames) < num_frames:
        return None

    video_np = np.array(frames)  # (T, H, W, C)
    return video_np



In [115]:
import os
target_dir = '/content/drive/MyDrive/LipertyData/LipertyData/datasets/word_dataset'

def get_dir_size(path):
    total = 0
    try:
        for root, dirs, files in os.walk(path):
            for f in files:
                fp = os.path.join(root, f)
                total += os.path.getsize(fp)
    except Exception:
        pass
    return total / (1024**3) # GB

if os.path.exists(target_dir):
    current_size = get_dir_size(target_dir)
    print('--- Extraction Progress ---')
    print(f'Current size on Drive: {current_size:.2f} GB')
    print(f'Target size: ~43 GB')
    print(f'Estimated completion: {(current_size/42.75)*100:.1f}%')

    # List first few folders to see structure
    !ls -F {target_dir} | head -n 5
else:
    print('Target directory not found yet.')

--- Extraction Progress ---
Current size on Drive: 15.95 GB
Target size: ~43 GB
Estimated completion: 37.3%
Lip Reading Dataset (word)/


In [106]:
import os
import psutil

# Check for any active python processes
print('--- Active Processes ---')
found_proc = False
for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
    try:
        cmd = ' '.join(proc.info['cmdline']) if proc.info['cmdline'] else ''
        if 'python' in proc.info['name'] and ('kaggle' in cmd or 'extract' in cmd):
            print(f'Active Process: PID {proc.info["pid"]} | {cmd}')
            found_proc = True
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

if not found_proc:
    print('No specific extraction process identified.')

# Inspect the '11' directory which usually contains the extracted files
version_path = '/root/.cache/kagglehub/datasets/lipreadingankaya/lip-reading-dataset-word/versions/11'
if os.path.exists(version_path):
    print(f'\n--- Contents of Version 11 Folder ---')
    contents = os.listdir(version_path)
    print(f'Items found: {len(contents)}')
    !du -sh {version_path}
    !ls -F {version_path} | head -n 10
else:
    print(f'\nVersion path {version_path} not found.')

--- Active Processes ---
No specific extraction process identified.

--- Contents of Version 11 Folder ---
Items found: 1
14G	/root/.cache/kagglehub/datasets/lipreadingankaya/lip-reading-dataset-word/versions/11
Lip Reading Dataset (word)/


In [103]:
import os
archive_path = '/root/.cache/kagglehub/datasets/lipreadingankaya/lip-reading-dataset-word/11.archive'
if os.path.exists(archive_path):
    size_gb = os.path.getsize(archive_path) / (1024**3)
    print(f'Current Archive Size: {size_gb:.2f} GB')
else:
    print('Archive file not found or already moved for extraction.')

Current Archive Size: 42.75 GB


In [107]:
import os
import subprocess

archive_path = '/root/.cache/kagglehub/datasets/lipreadingankaya/lip-reading-dataset-word/11.archive'
target_dir = '/content/drive/MyDrive/LipertyData/LipertyData/datasets/word_dataset'
os.makedirs(target_dir, exist_ok=True)

print(f'--- Starting Manual Extraction to Drive ---')
print(f'Source: {archive_path}')
print(f'Target: {target_dir}')

# Using a subprocess to extract directly to drive to bypass local storage limits
# We'll use 'unzip' since it's common for Kaggle archives, but we'll check first
if os.path.exists(archive_path):
    # Extracting while ignoring the top-level folder to flatten structure if needed
    cmd = f'unzip -q {archive_path} -d {target_dir}'
    print('Executing extraction... (This may take a while)')
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print(f'Extraction started in background. Monitor {target_dir} for new files.')
else:
    print('Archive file not found.')

--- Starting Manual Extraction to Drive ---
Source: /root/.cache/kagglehub/datasets/lipreadingankaya/lip-reading-dataset-word/11.archive
Target: /content/drive/MyDrive/LipertyData/LipertyData/datasets/word_dataset
Executing extraction... (This may take a while)
Extraction started in background. Monitor /content/drive/MyDrive/LipertyData/LipertyData/datasets/word_dataset for new files.


In [102]:
import os

# Check locations for the downloaded Kaggle datasets
kaggle_root = '/root/.cache/kagglehub/datasets'

print('--- Locating Kaggle Datasets ---')
if os.path.exists(kaggle_root):
    for author in os.listdir(kaggle_root):
        author_path = os.path.join(kaggle_root, author)
        for dataset_name in os.listdir(author_path):
            full_path = os.path.join(author_path, dataset_name)
            print(f'Found dataset: {dataset_name} at {full_path}')
            # List subdirectories to understand structure
            !ls -F {full_path}
else:
    print('Kagglehub cache directory not found.')

--- Locating Kaggle Datasets ---
Found dataset: lip-reading-dataset-word at /root/.cache/kagglehub/datasets/lipreadingankaya/lip-reading-dataset-word
11.archive  versions/
Found dataset: grid-lip-reading-s10-preprocessed3 at /root/.cache/kagglehub/datasets/awatefchiha/grid-lip-reading-s10-preprocessed3
1.complete  versions/


In [94]:
import numpy as np
import os

# Inspecting a sample GRID .npy file to understand its dimensions
sample_grid = '/root/.cache/kagglehub/datasets/awatefchiha/grid-lip-reading-s10-preprocessed3/versions/1/now/now_0018.npy'
if os.path.exists(sample_grid):
    data = np.load(sample_grid)
    print(f'GRID .npy shape: {data.shape}')
    print(f'Data type: {data.dtype}')

# Checking if the TF model weights can be partially read to see layer names
# This gives a hint about the model architecture (e.g., GRU, LSTM, or Transformer)
try:
    import h5py
    with h5py.File('/root/.cache/kagglehub/models/shoshinmai/lip-read-2.0/tensorFlow2/default/3/models/checkpoint.weights.h5', 'r') as f:
        print('\n--- TF Model Layer Names ---')
        for key in f.keys():
            print(key)
except Exception as e:
    print(f'Could not read H5 keys: {e}')

GRID .npy shape: (10, 64, 64, 3)
Data type: float32

--- TF Model Layer Names ---
layers
optimizer
vars


In [54]:
!ls -R /content/Liperty/data/GRID

/content/Liperty/data/GRID:
unknown

/content/Liperty/data/GRID/unknown:
train

/content/Liperty/data/GRID/unknown/train:


In [52]:
import os
import glob

grid_path = '/content/Liperty/data/GRID'
files = glob.glob(os.path.join(grid_path, '**/*.mp4'), recursive=True)

print(f'--- GRID Data Check ---')
print(f'Total .mp4 files found: {len(files)}')
if len(files) > 0:
    print('First 5 files:')
    for f in files[:5]:
        print(f'- {f}')
else:
    print('No videos found. We may need to troubleshoot the Hugging Face download logic.')

--- GRID Data Check ---
Total .mp4 files found: 0
No videos found. We may need to troubleshoot the Hugging Face download logic.


In [96]:
# Training on GRID dataset
VIDEOS_ROOT = "/content/Liperty/data/GRID"
SAVE_PATH = "/content/drive/MyDrive/LipertyData/LipertyData/checkpoints/best_model.pth"

import os
cmd = f"""export PYTHONPATH=$PYTHONPATH:/content/Liperty/VALLR && \
export WANDB_MODE=offline && \
python -u /content/Liperty/VALLR/main.py \
    --mode train \
    --videos_root {VIDEOS_ROOT} \
    --save_model_path {SAVE_PATH} \
    --batch_size 2 \
    --epochs 21 \
    --num_workers 1"""

print("Starting training on GRID subset...")
get_ipython().system(cmd)

Starting training on GRID subset...
Training
Model has 88,029,736 trainable parameters.
Loaded 50 videos from train split in /content/Liperty/data/GRID.
Traceback (most recent call last):
  File "/content/Liperty/VALLR/main.py", line 586, in <module>
    main(args)
  File "/content/Liperty/VALLR/main.py", line 581, in main
    train(device, version, video_path, batch_size, num_workers, epochs, save_model_path, sample_size, vocab)
  File "/content/Liperty/VALLR/main.py", line 357, in train
    validation_dataset = VideoDataset(
                         ^^^^^^^^^^^^^
  File "/content/Liperty/VALLR/Data/dataset.py", line 37, in __init__
    raise ValueError(f"No video files found in directory: {video_dir} for split: {split}")
ValueError: No video files found in directory: /content/Liperty/data/GRID for split: val


In [112]:
# Cell removed - GRID data preparation no longer needed

In [43]:
import os
# Search for GRID related files in the repo
!find /content/Liperty -iname "*grid*"

/content/Liperty/app/src/main/cpp/libs/opencv/sdk/java/src/org/opencv/objdetect/GridBoard.java
/content/Liperty/app/src/main/cpp/libs/opencv/sdk/java/src/org/opencv/ml/ParamGrid.java
/content/Liperty/app/src/main/cpp/libs/opencv/sdk/java/javadoc/org/opencv/objdetect/GridBoard.html
/content/Liperty/app/src/main/cpp/libs/opencv/sdk/java/javadoc/org/opencv/ml/ParamGrid.html


In [49]:
import os

# Read the specific section of fetch_datasets.py related to GRID
print("--- Extracting GRID logic from fetch_datasets.py ---")
!grep -A 20 "GRID Dataset" /content/Liperty/tools/fetch_datasets.py

# Also check if there's a specific requirements file for the tools
if os.path.exists('/content/Liperty/VALLR/requirements.txt'):
    print("\n--- VALLR Requirements ---")
    !cat /content/Liperty/VALLR/requirements.txt

--- Extracting GRID logic from fetch_datasets.py ---
    print("\n[6] GRID Dataset (Hugging Face Subset)")
    print("    Package: datasets (pip install datasets)")
    print("    Command: from datasets import load_dataset; ds = load_dataset('wissemkarous/lipreading')")
    print(f"    Local Backup Path: {grid_dir}")
    
    print("\n========================================")
    print("Next Steps:")
    print("1. Obtain the datasets manually using the links above.")
    print("2. Place them in the created 'data/' directories.")
    print("3. Use 'tools/external/auto_avsr/preprocessing' scripts to crop/process video.")
    print("4. Use 'tools/create_trainable_model.py' to generate a LoRA-ready model.")
    print("========================================")

if __name__ == "__main__":
    setup_datasets()

--- VALLR Requirements ---
absl-py==2.1.0
attrs==24.2.0
audioread==3.0.1
certifi==2024.7.4
cffi==1.17.0
charset-normalizer==3.3.2
click==8.1.7
cmudict==1.0.31
ConfigArgParse==1.7
con

In [35]:
import sys
import os

# Add repo to path
repo_path = '/content/Liperty/VALLR'
if repo_path not in sys.path:
    sys.path.append(repo_path)

print('--- Checking config.py for arguments ---')
config_path = os.path.join(repo_path, 'config.py')
if os.path.exists(config_path):
    !head -n 50 {config_path}

print('\n--- Manual Import & Initialization Test ---')
try:
    # Set dummy env vars for WandB to prevent blocking
    os.environ['WANDB_MODE'] = 'offline'

    import torch
    from Models.ML_VALLR import VideoViTMasked as ML_VALLR
    from config import load_args
    print('Core imports successful.')

    # Test if load_args() fails with current sys.argv
    # We simulate the CLI args here
    sys.argv = ['main.py', '--videos_root', '/content/Liperty/data/VVAD-LRS3', '--save_model_path', '/content/test.pth']
    args = load_args()
    print('Argument loading successful.')

except Exception as e:
    print(f'CAUGHT ERROR: {e}')
    import traceback
    traceback.print_exc()

--- Checking config.py for arguments ---
import configargparse
import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau
import csv
import numpy as np

def load_args():
    parser = configargparse.ArgumentParser(description="Main")

    # Device configuration
    parser.add_argument('--device', type=str, default='cuda', help="Device to use for computation ('cuda' or 'cpu')")

    # Model configuration
    # parser.add_argument('--ckpt_path', default=None, help="Path to the model checkpoint for loading")
    parser.add_argument('--save_model_path', type=str, help="Path to save the trained model")
    # parser.add_argument('--model_path', type=str, help="Path to the trained model for inference")
    parser.add_argument('--version', type=str, choices=['V1', 'V2'], default='V1', help="Model version to use: 'V1' or 'V2'")

    # Data configuration
    # parser.add_argument('--feat_dim', type=int, default=512, help="Video features dimension (used if loading features directly)")
   

In [34]:
import os
# Check if the file exists and read the header/imports
script_path = '/content/Liperty/VALLR/main.py'
if os.path.exists(script_path):
    print(f'--- Header of {script_path} ---')
    !head -n 50 {script_path}
else:
    print('Script not found at expected path.')

# Try to run a simple syntax check
print('\n--- Syntax Check ---')
!python -m py_compile /content/Liperty/VALLR/main.py

--- Header of /content/Liperty/VALLR/main.py ---
from curses import version
import torch
import wandb
from tqdm import tqdm
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.nn.utils.rnn as rnn_utils
from Models.ML_VALLR import VideoViTMasked as ML_VALLR
from Models.VALLR import VALLR
from Data.dataset import VideoDataset
import torch.nn as nn
from config import load_args, WarmupScheduler, get_vocab
import numpy as np
from transformers import VideoMAEConfig, Wav2Vec2Config, Wav2Vec2Processor, GPT2Tokenizer, GPT2LMHeadModel
import os
import numpy as np
from decord import VideoReader, cpu
import torch.nn.functional as F
import jiwer
from torch.amp import autocast, GradScaler
from itertools import zip_longest

args = load_args()

def monitor_gradients(model):
    total_norm = 0
    for name, param in model.named_parameters():
        if param.grad is not None:
            param_norm = param.grad.norm(2)
            total_norm += pa

In [70]:
import psutil
import torch
import os

# 1. Check for active main.py process
found = False
for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
    try:
        if proc.info['cmdline'] and 'main.py' in ' '.join(proc.info['cmdline']):
            print(f'Active Training Process: PID {proc.info["pid"]}')
            found = True
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

if not found:
    print('No active training process detected. Ready for restart.')

# 2. Check GPU
if torch.cuda.is_available():
    print(f'GPU VRAM Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB')


No active training process detected. Ready for restart.
GPU VRAM Allocated: 0.00 MB


In [31]:
import psutil
import torch

# Check GPU
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print(f"Cached: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")
else:
    print("No GPU detected.")

# Check System CPU/RAM
print(f"RAM Usage: {psutil.virtual_memory().percent}%")
print(f"CPU Usage: {psutil.cpu_percent()}%")

GPU: Tesla T4
Allocated: 0.00 MB
Cached: 0.00 MB
RAM Usage: 13.3%
CPU Usage: 39.5%


In [28]:
import psutil
import os

# Check for running python processes related to main.py
found = False
for proc in psutil.process_iter(['pid', 'name', 'cmdline']):
    try:
        if proc.info['cmdline'] and 'main.py' in ' '.join(proc.info['cmdline']):
            print(f"Process Found: PID {proc.info['pid']} | Cmd: {' '.join(proc.info['cmdline'])}")
            found = True
    except (psutil.NoSuchProcess, psutil.AccessDenied, psutil.ZombieProcess):
        pass

if not found:
    print("No active training process (main.py) detected.")

No active training process (main.py) detected.


In [79]:
# Improving the patch to avoid RecursionError and skipping invalid samples robustly
import os

dataset_file = '/content/Liperty/VALLR/Data/dataset.py'
with open(dataset_file, 'r') as f:
    lines = f.readlines()

new_lines = []
for line in lines:
    # We check for the specific lines that cause recursive depth issues
    if 'raise ValueError(f"Exceeded max retries' in line or 'return self.__getitem__' in line:
        indent = line[:line.find(line.strip()[0])]
        # Use a literal string to ensure {idx} is written to the file correctly
        new_lines.append(indent + "print(f'Warning: Skipping invalid sample at index {idx}')\n")
        new_lines.append(indent + "return None  # Return None to signal failure without recursion\n")
    else:
        new_lines.append(line)

with open(dataset_file, 'w') as f:
    f.writelines(new_lines)

# Also patch main.py to handle 'None' from dataset using a filter in the dataloader or loop
main_file = '/content/Liperty/VALLR/main.py'
# Ensure we don't double-patch if run twice
!sed -i 's/for batch in tqdm(dataloader/for batch in tqdm(filter(lambda x: x is not None, dataloader)/g' {main_file}

print("Applied non-recursive skipping patch to dataset.py and main.py.")

Applied non-recursive skipping patch to dataset.py and main.py.


## Export to TFLite

In [ ]:
!python tools/convert_vallr.py